# Simplified path — Colab GPU run

Runs `src/simplified.py` end-to-end on a Colab GPU using a larger model than the local Qwen 1.5B default. Same code as local; only the model size changes.

**Before running:** Runtime → Change runtime type → GPU (T4 is enough for 7B in fp16).

Outputs are written to `colab_outputs.md` in the working directory; download from the Files panel after the run.

## 1. Clone repo & install dependencies

In [ ]:
# Cloning from b-coman fork colab-refactor branch for Colab testing.
# Once the PR is merged into dariacoman/main, switch this clone URL.
!git clone --branch colab-refactor https://github.com/b-coman/compliance-gap-analysis.git
%cd compliance-gap-analysis

In [ ]:
# Install only what Colab does not preinstall.
# Colab already ships torch, numpy, etc. — installing requirements.txt
# would downgrade Colab torch and break the CUDA wheel alignment.
!pip install -q sentence-transformers transformers accelerate diskcache

## 2. Pick the model

`MODEL_ID` is read by `src/simplified.py` at import time. Set it before importing.

Recommended starting points:
- `Qwen/Qwen2.5-7B-Instruct` — fits T4 in fp16 (~14 GB), no HF gating
- `Qwen/Qwen2.5-1.5B-Instruct` — local default, useful for sanity-checking parity with local
- `google/gemma-2-2b-it` — requires HF auth + license acceptance (gated)

In [ ]:
import os
os.environ['MODEL_ID'] = 'Qwen/Qwen2.5-7B-Instruct'
os.environ['PYTHONPATH'] = '/content/compliance-gap-analysis'

## 3. Verify GPU & import the simplified path

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
    print('Memory:', torch.cuda.get_device_properties(0).total_memory / 1e9, 'GB')

In [ ]:
from src.simplified import analyse, LLM_MODEL_ID
print('Model in use:', LLM_MODEL_ID)

## 4. Run the 5 standard test queries

Verbatim text from `docs/test-queries.md`. These are the same queries run in the local baseline (`docs/test-passes/v3-qwen-1.5b-local-baseline.md`) — outputs from this notebook are directly comparable.

In [ ]:
QUERIES = {
    'Q1 multi-facet': (
        "TalentLens compliance under EU AI Act Annex III \u00a74 \u2014 am I covered on "
        "Article 13 deployer instructions, Article 14 human oversight, Article 26 logs "
        "and worker information, and the related Article 22 GDPR automated-decisions "
        "duties? Where are the gaps?"
    ),
    'Q2 red-teaming': (
        "Does our policy address the red-teaming requirements before deploying a high-"
        "risk AI system to production?"
    ),
    'Q3 Article 22 sub-clauses': (
        "How do we meet GDPR Article 22 requirements on solely automated decisions "
        "affecting candidates \u2014 explicit consent, right to obtain human intervention, "
        "right to contest the decision, and right to express their point of view?"
    ),
    'Q4 transparency': (
        "Are we doing enough on transparency for candidates assessed by TalentLens?"
    ),
    'Q5 FRIA': (
        "Have we performed a Fundamental Rights Impact Assessment under EU AI Act "
        "Article 27 for TalentLens as a deployer of an Annex III high-risk system?"
    ),
}

In [ ]:
import time

results = {}
for label, query in QUERIES.items():
    print(f'\n{"="*70}\n{label}\n{"="*70}')
    print(f'Query: {query}\n')
    t0 = time.time()
    output = analyse(query)
    elapsed = time.time() - t0
    results[label] = {'query': query, 'output': output, 'elapsed_s': elapsed}
    print(output)
    print(f'\n[{elapsed:.1f}s]')

## 5. Save outputs to a markdown file

Download `colab_outputs.md` from the Files panel afterwards. Drop it into `docs/test-passes/` in the repo with a descriptive filename, e.g. `v4-qwen-7b-colab.md`.

In [ ]:
from datetime import datetime, timezone

lines = [
    f'# Test pass \u2014 simplified path on Colab GPU',
    '',
    f'> Run date: {datetime.now(timezone.utc).strftime("%Y-%m-%d")}',
    f'> Model: `{LLM_MODEL_ID}`',
    f'> Hardware: Colab GPU ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"})',
    f'> Prompt: V4 (current default in `src/simplified.py`)',
    f'> Embeddings: BAAI/bge-large-en-v1.5',
    '',
    '---',
    '',
]
for label, r in results.items():
    lines += [
        f'## {label}',
        '',
        f'**Query:** {r["query"]}',
        '',
        f'**Generation latency:** {r["elapsed_s"]:.1f}s',
        '',
        '**Output:**',
        '',
        '```',
        r['output'].rstrip(),
        '```',
        '',
        '---',
        '',
    ]

with open('colab_outputs.md', 'w') as f:
    f.write('\n'.join(lines))

print('Wrote colab_outputs.md')